# Data Preparation: Alamedin & Ala-Archa Basins

Preprocessing of meteorological forcing, discharge observations, and catchment
descriptors for two glaciated basins in the Tien Shan (Kyrgyzstan). Produces
`Forcing`, `StreamflowSeries`, and `Catchment` objects ready for
GR6J+CemaNeige+Glacier calibration.

In [ ]:
from pathlib import Path
from typing import NamedTuple

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import rasterio.features
import seaborn as sns

from wat_mod_giz import Catchment, Forcing, StreamflowSeries
from wat_mod_giz.calibration.metrics import get_metric, kge, nse
from wat_mod_giz.models import gr6j_cemaneige_glacier

sns.set_context("paper", font_scale=1.3)


class BasinConfig(NamedTuple):
    name: str
    folder: str
    area_km2: float
    rgi_file: str
    gauge_code: int
    gauge_lat: float


BASINS = {
    "15189": BasinConfig(
        name="Alamedin",
        folder="15189_Alamedin_River_KGZ",
        area_km2=414.05,
        rgi_file="RGI_Alamedin.shp",
        gauge_code=15189,
        gauge_lat=42.697,
    ),
    "15194": BasinConfig(
        name="Ala-Archa",
        folder="15194_AlaArcha_River_KGZ",
        area_km2=270.84,
        rgi_file="RGI_Ala-Archa.shp",
        gauge_code=15194,
        gauge_lat=42.660,
    ),
}

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data" / "02_data"
N_LAYERS = 3
WARMUP = 365
SHARED_GAUGES_FILENAME = "Ala-Archa_Alamedin_gauges.shp"

BASIN_LONG_STREAMFLOW_FILENAMES = {
    "15189": "discharge_data_long_alamedin.xlsx",
    "15194": "discharge_data_long_Ala-Archa.xlsx",
}
BASIN_DEFAULT_PERIOD_DEFINITIONS = {
    "15189": {
        "validation": (pd.Timestamp("2000-01-01"), pd.Timestamp("2005-12-31")),
        "calibration": (pd.Timestamp("2010-01-01"), pd.Timestamp("2019-12-31")),
    },
    "15194": {
        "validation": (pd.Timestamp("2000-01-01"), pd.Timestamp("2004-12-31")),
        "calibration": (pd.Timestamp("2010-01-01"), pd.Timestamp("2019-12-31")),
    },
}

TARGET_BASIN_CODE = "15194"  # "15189" = Alamedin, "15194" = Ala-Archa
SHORT_GAP_LIMIT_DAYS = 5
TARGET_PERIOD_DEFINITIONS = BASIN_DEFAULT_PERIOD_DEFINITIONS[TARGET_BASIN_CODE].copy()
CALIBRATION_SETTINGS = {
    "objective": "nse",
    "warmup": WARMUP,
    "population_size": 30,
    "generations": 20,
    "seed": 42,
    "progress": False,
    "return_diagnostics": False,
    "n_workers": -1,
}


def resolve_shared_gis_root() -> Path:
    """Find the shared GIS folder that contains the common gauges shapefile."""
    for cfg in BASINS.values():
        candidate = DATA_ROOT / cfg.folder / "00_gis"
        if (candidate / SHARED_GAUGES_FILENAME).exists():
            return candidate
    raise FileNotFoundError(f"Could not find {SHARED_GAUGES_FILENAME} under any basin GIS folder.")


SHARED_GIS_ROOT = resolve_shared_gis_root()


## Potential Evapotranspiration — Oudin (2005)

A temperature-only PET estimate requiring just mean daily temperature $T$ in $^\circ\mathrm{C}$, day of year $J$, and station latitude $\phi$ in radians.

### Extraterrestrial radiation $R_a$

| Quantity | Formula |
|---|---|
| Solar declination | $\delta = 0.4093 \sin\left(\frac{2\pi J}{365} - 1.405\right)$ |
| Sunset hour angle | $\omega_s = \arccos(-\tan\phi\,\tan\delta)$ |
| Inverse relative distance | $d_r = 1 + 0.033\cos\left(\frac{2\pi J}{365}\right)$ |

$$
R_a = \frac{24 \times 60}{\pi}\, G_{sc}\, d_r
      \left[\omega_s \sin\phi\,\sin\delta
            + \cos\phi\,\cos\delta\,\sin\omega_s\right]
$$

where $G_{sc} = 0.0820\,\mathrm{MJ\,m^{-2}\,min^{-1}}$.

### PET

$$
\operatorname{PET} =
\begin{cases}
\frac{R_a}{\lambda}\,\frac{T + 5}{100}
  & \text{if } T + 5 > 0 \\[6pt]
0 & \text{otherwise}
\end{cases}
$$

with $\lambda = 2.45\,\mathrm{MJ\,kg^{-1}}$ (latent heat of vaporisation).

In [ ]:
GSC = 0.0820  # solar constant [MJ m⁻² min⁻¹]
LAMBDA = 2.45  # latent heat of vaporisation [MJ kg⁻¹]


def extraterrestrial_radiation(doy: np.ndarray, lat_rad: float) -> np.ndarray:
    """Daily extraterrestrial radiation R_a [MJ m⁻² day⁻¹] (FAO-56)."""
    dr = 1.0 + 0.033 * np.cos(2.0 * np.pi / 365.0 * doy)
    delta = 0.4093 * np.sin(2.0 * np.pi / 365.0 * doy - 1.405)
    ws = np.arccos(-np.tan(lat_rad) * np.tan(delta))
    return (
        (24.0 * 60.0 / np.pi)
        * GSC
        * dr
        * (ws * np.sin(lat_rad) * np.sin(delta) + np.cos(lat_rad) * np.cos(delta) * np.sin(ws))
    )


def oudin_pet(temp: np.ndarray, doy: np.ndarray, latitude_deg: float) -> np.ndarray:
    """Oudin (2005) PET [mm day⁻¹] from mean daily temperature."""
    lat_rad = np.radians(latitude_deg)
    ra = extraterrestrial_radiation(doy, lat_rad)
    return np.where(temp + 5.0 > 0.0, ra / LAMBDA * (temp + 5.0) / 100.0, 0.0)

## Solid Precipitation Partitioning

CemaNeige requires the mean annual solid precipitation $P_s$ as a catchment descriptor. Solid fraction is estimated with the USACE linear rain/snow threshold:

$$
f_s =
\begin{cases}
1, & T \leq T_s \\
\frac{T_r - T}{T_r - T_s}, & T_s < T < T_r \\
0, & T \geq T_r
\end{cases}
$$

with $T_s = -1^\circ\mathrm{C}$ and $T_r = 3^\circ\mathrm{C}$.

Mean annual solid precipitation ($\mathrm{mm\,yr^{-1}}$):

$$
P_s = 365.25 \times \frac{1}{N}\sum_{i=1}^{N} f_{s,i} P_i
$$

In [ ]:
T_SNOW = -1.0  # all-snow threshold [°C]
T_RAIN = 3.0  # all-rain threshold [°C]


def solid_fraction(temp: np.ndarray) -> np.ndarray:
    """USACE linear rain/snow partitioning."""
    return np.clip((T_RAIN - temp) / (T_RAIN - T_SNOW), 0.0, 1.0)


def mean_annual_solid_precip(precip: np.ndarray, temp: np.ndarray) -> float:
    """Mean annual solid precipitation [mm yr⁻¹]."""
    daily_solid = precip * solid_fraction(temp)
    return float(np.mean(daily_solid) * 365.25)

## Load & Process Forcing Data

**Data sources:** ERA5-Land daily basin-averaged precipitation ($\mathrm{mm\,day^{-1}}$) and temperature ($^\circ\mathrm{C}$), stored as CSVs with a `date` index column.

**Discharge conversion** from volumetric to specific:

$$
Q_{\mathrm{mm\,day^{-1}}} = Q_{\mathrm{m^3\,s^{-1}}} \times \frac{86.4}{A_{\mathrm{km^2}}}
$$

(factor $86.4 = 86\,400\,\mathrm{s\,day^{-1}} / 1\,000\,\mathrm{mm\,m^{-1}} \times 1\,\mathrm{km^2} / 10^6\,\mathrm{m^2}$
simplifies to $86\,400 / 10^6 \times 10^3 = 0.0864 \times 10^3$).

**Strategy:**
1. Build continuous daily forcing from ERA5 for each basin.
2. Choose the active basin in `TARGET_BASIN_CODE` and edit `TARGET_PERIOD_DEFINITIONS` to set the train/validation dates.
3. For the active basin, read the moved long workbook, interpolate only internal gaps up to `SHORT_GAP_LIMIT_DAYS`, and preserve longer gaps as real breaks.
4. For the inactive basin, keep the old longest contiguous observed block so the descriptive basin setup still runs.
5. Extend forcing backward by the warmup period so the model state can equilibrate before each scored window begins.


In [ ]:
def longest_contiguous_run(dates: pd.DatetimeIndex) -> slice:
    """Return a slice for the longest run of consecutive daily dates."""
    gaps = dates.to_series().diff().dt.days.ne(1)
    group_ids = gaps.cumsum()
    group_sizes = group_ids.value_counts()
    best_group = group_sizes.idxmax()
    mask = group_ids == best_group
    idx = mask.values.nonzero()[0]
    return slice(idx[0], idx[-1] + 1)


def compute_gap_lengths(series: pd.Series) -> pd.Series:
    """Return the size of each missing-data gap for every row in the series."""
    missing = series.isna()
    group_ids = missing.ne(missing.shift(fill_value=False)).cumsum()
    gap_lengths = missing.groupby(group_ids).transform("sum")
    return gap_lengths.where(missing, 0).astype(int)


def summarize_missing_gaps(dates: pd.Series, values: pd.Series) -> pd.DataFrame:
    """Summarize contiguous missing periods in a daily timeseries."""
    missing_dates = dates.loc[values.isna()]
    if missing_dates.empty:
        return pd.DataFrame(columns=["start", "end", "size"])

    group_ids = missing_dates.diff().dt.days.ne(1).cumsum()
    return (
        missing_dates.groupby(group_ids)
        .agg(start="min", end="max", size="size")
        .sort_values("size", ascending=False)
        .reset_index(drop=True)
    )


def build_period(
    *,
    era5: pd.DataFrame,
    q_df: pd.DataFrame,
    obs_start: pd.Timestamp,
    obs_end: pd.Timestamp,
    warmup_days: int,
) -> dict[str, object]:
    """Build one continuous forcing/observation block with warmup."""
    forcing_start = obs_start - pd.Timedelta(days=warmup_days)
    forcing_mask = era5["date"].between(forcing_start, obs_end)
    forcing_df = era5.loc[forcing_mask].reset_index(drop=True)

    obs_mask = q_df["date"].between(obs_start, obs_end)
    observed_df = q_df.loc[obs_mask, ["date", "q_mm"]].sort_values("date").reset_index(drop=True)
    if observed_df["q_mm"].isna().any():
        raise ValueError(f"Observed streamflow contains NaN values between {obs_start.date()} and {obs_end.date()}.")

    forcing = Forcing(
        time=forcing_df["date"].values.astype("datetime64[D]"),
        precip=forcing_df["precipitation_mm"].values,
        pet=forcing_df["pet_mm"].values,
        temp=forcing_df["temperature_degC"].values,
    )
    observed = StreamflowSeries(
        time=forcing.time[warmup_days:],
        streamflow=observed_df["q_mm"].values,
    )
    return {
        "forcing": forcing,
        "observed": observed,
        "obs_start": obs_start,
        "obs_end": obs_end,
        "warmup_start": forcing_start,
        "warmup_days": warmup_days,
    }


def load_long_streamflow(
    cfg: BasinConfig,
    streamflow_path: Path,
    short_gap_limit_days: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load a long workbook and preserve gaps longer than the interpolation limit."""
    q_df = pd.read_excel(streamflow_path)
    q_df["date"] = pd.to_datetime(
        {
            "year": q_df["year"].astype(int),
            "month": q_df["month"].astype(int),
            "day": q_df["day"].astype(int),
        }
    )
    q_df = (
        q_df[["date", "discharge_m3_per_sec"]]
        .rename(columns={"discharge_m3_per_sec": "q_m3s_raw"})
        .sort_values("date")
        .drop_duplicates("date")
        .reset_index(drop=True)
    )

    full_dates = pd.date_range(q_df["date"].min(), q_df["date"].max(), freq="D")
    q_df = q_df.set_index("date").reindex(full_dates).rename_axis("date").reset_index()
    q_df["gap_length_days"] = compute_gap_lengths(q_df["q_m3s_raw"])

    interpolated = q_df["q_m3s_raw"].interpolate(method="linear", limit_area="inside")
    short_gap_mask = q_df["gap_length_days"].between(1, short_gap_limit_days)
    q_df["q_m3s"] = q_df["q_m3s_raw"]
    q_df.loc[short_gap_mask, "q_m3s"] = interpolated.loc[short_gap_mask]
    q_df["q_status"] = "observed"
    q_df.loc[short_gap_mask, "q_status"] = f"interpolated_1to{short_gap_limit_days}d_gap"
    q_df.loc[q_df["q_m3s"].isna(), "q_status"] = "missing_long_gap"
    q_df["q_mm"] = q_df["q_m3s"] * 86.4 / cfg.area_km2

    gap_summary = summarize_missing_gaps(q_df["date"], q_df["q_m3s"])
    return q_df, gap_summary


if TARGET_BASIN_CODE not in BASINS:
    raise ValueError(f"Unknown TARGET_BASIN_CODE: {TARGET_BASIN_CODE}")

basin_data: dict[str, dict[str, object]] = {}

for code, cfg in BASINS.items():
    base = DATA_ROOT / cfg.folder

    precip_df = pd.read_csv(base / "02_forcing" / "era5_precipitation_daily.csv", parse_dates=["date"])
    temp_df = pd.read_csv(base / "02_forcing" / "era5_temperature_daily.csv", parse_dates=["date"])
    era5 = precip_df.merge(temp_df, on="date")
    era5["pet_mm"] = oudin_pet(era5["temperature_degC"].values, era5["date"].dt.dayofyear.values, cfg.gauge_lat)

    masp = mean_annual_solid_precip(era5["precipitation_mm"].values, era5["temperature_degC"].values)
    basin_data[code] = {
        "cfg": cfg,
        "masp": masp,
        "periods": {},
    }

    if code == TARGET_BASIN_CODE:
        streamflow_path = base / "01_gauge_data" / BASIN_LONG_STREAMFLOW_FILENAMES[code]
        q_df, gap_summary = load_long_streamflow(cfg, streamflow_path, SHORT_GAP_LIMIT_DAYS)
        basin_data[code]["streamflow_source"] = streamflow_path
        basin_data[code]["streamflow_df"] = q_df

        short_gap_days = int(q_df["q_status"].eq(f"interpolated_1to{SHORT_GAP_LIMIT_DAYS}d_gap").sum())
        print(
            f"{cfg.name}: workbook {streamflow_path.name}, interpolated short-gap days = {short_gap_days}, "
            f"remaining long-gap days = {int(q_df['q_m3s'].isna().sum())}"
        )
        if gap_summary.empty:
            print(f"  {cfg.name}: no preserved long gaps remain after short-gap interpolation.")
        else:
            largest_gap = gap_summary.iloc[0]
            print(
                f"  largest preserved gap = {largest_gap['start'].date()} to {largest_gap['end'].date()} "
                f"({int(largest_gap['size'])} days)"
            )

        period_defs = TARGET_PERIOD_DEFINITIONS
    else:
        streamflow_path = base / "01_gauge_data" / BASIN_LONG_STREAMFLOW_FILENAMES[code]
        q_df, gap_summary = load_long_streamflow(cfg, streamflow_path, SHORT_GAP_LIMIT_DAYS)
        basin_data[code]["streamflow_source"] = streamflow_path
        basin_data[code]["streamflow_df"] = q_df

        available_dates = pd.DatetimeIndex(q_df.loc[q_df["q_mm"].notna(), "date"])
        overlap = pd.DatetimeIndex(era5["date"]).intersection(available_dates)
        run = longest_contiguous_run(overlap.sort_values())
        obs_dates = overlap.sort_values()[run]
        period_defs = {"longest_run": (obs_dates[0], obs_dates[-1])}

    for period_name, (obs_start, obs_end) in period_defs.items():
        period = build_period(
            era5=era5,
            q_df=q_df,
            obs_start=obs_start,
            obs_end=obs_end,
            warmup_days=WARMUP,
        )
        basin_data[code]["periods"][period_name] = period
        print(
            f"  {cfg.name} [{period_name}]: forcing {period['warmup_start'].date()}–{period['obs_end'].date()} "
            f"({len(period['forcing'])} days), observed {period['obs_start'].date()}–{period['obs_end'].date()} "
            f"({len(period['observed'])} days), mean annual solid precip = {masp:.0f} mm/yr"
        )


## Hypsometric Curve

A 101-element array giving the elevation (m a.s.l.) below which each percentile
(0 %, 1 %, ..., 100 %) of the catchment area lies. CemaNeige uses the
hypsometric curve to partition the catchment into $N$ elevation bands with
equal area, each receiving its own snow and glacier accounting.

**Computation:** extract all DEM pixel values inside the catchment boundary,
then take `np.percentile(elevations, range(101))`.

In [ ]:
hypsometric_curves: dict[str, np.ndarray] = {}

for code, cfg in BASINS.items():
    dem_path = DATA_ROOT / cfg.folder / "02_forcing" / "dem.tif"
    with rasterio.open(dem_path) as src:
        dem = src.read(1)
        valid = dem[~np.isnan(dem)]

    curve = np.percentile(valid, np.arange(101))
    hypsometric_curves[code] = curve
    print(f"{cfg.name}: elevation {curve[0]:.0f}–{curve[-1]:.0f} m a.s.l. (median {curve[50]:.0f} m)")

In [ ]:
sns.set_context("paper", font_scale=1.3)

fig, ax = plt.subplots(figsize=(7, 7))
percentiles = np.arange(101)

for code, cfg in BASINS.items():
    sns.lineplot(x=percentiles, y=hypsometric_curves[code], ax=ax, label=cfg.name, linewidth=2)
    ax.grid(True, which="both", linestyle="--", alpha=0.3)

ax.set_xlabel("Catchment area percentile [%]")
ax.set_ylabel("Elevation [m a.s.l.]")
sns.despine()
plt.show()


## Gauge Elevation

`input_elevation` is the elevation (m a.s.l.) of the meteorological reference
point — here the gauging station. It is sampled from the DEM at the gauge
coordinates and serves as the anchor for temperature and precipitation lapse-rate
extrapolation across elevation bands.

In [ ]:
gauges = gpd.read_file(SHARED_GIS_ROOT / SHARED_GAUGES_FILENAME)
gauge_elevations: dict[str, float] = {}

for code, cfg in BASINS.items():
    gauge = gauges.loc[gauges["CODE"] == str(cfg.gauge_code)].iloc[0]
    lon, lat = gauge.geometry.x, gauge.geometry.y

    dem_path = DATA_ROOT / cfg.folder / "02_forcing" / "dem.tif"
    with rasterio.open(dem_path) as src:
        dem = src.read(1)
        row, col = src.index(lon, lat)
        elev = dem[row, col]

        # Gauge at outlet may fall outside valid DEM — use nearest valid pixel
        if np.isnan(elev):
            valid_mask = ~np.isnan(dem)
            rows, cols = np.where(valid_mask)
            dists = (rows - row) ** 2 + (cols - col) ** 2
            nearest = dists.argmin()
            elev = dem[rows[nearest], cols[nearest]]

    gauge_elevations[code] = float(elev)
    print(f"{cfg.name} gauge: ({lon:.3f}°E, {lat:.3f}°N) → {elev:.0f} m a.s.l.")


## Glacier Fractions per Elevation Band

Each of the $N$ elevation bands requires a glacier coverage fraction
$f_{\text{glacier}}^{(k)} \in [0, 1]$.

**Steps:**
1. Rasterize the RGI glacier outlines onto the DEM grid (binary mask).
2. Derive elevation-band boundaries from the hypsometric curve
   (equal-area quantiles).
3. For each band $k$, compute

$$
f_{\text{glacier}}^{(k)}
  = \frac{n_{\text{glacier}}^{(k)}}{n_{\text{total}}^{(k)}}
$$

where $n_{\text{glacier}}^{(k)}$ is the number of glacierised DEM pixels in
band $k$ and $n_{\text{total}}^{(k)}$ is the total pixel count in that band.

In [ ]:
glacier_fractions: dict[str, np.ndarray] = {}

for code, cfg in BASINS.items():
    rgi = gpd.read_file(SHARED_GIS_ROOT / cfg.rgi_file)

    dem_path = DATA_ROOT / cfg.folder / "02_forcing" / "dem.tif"
    with rasterio.open(dem_path) as src:
        dem = src.read(1)
        transform = src.transform
        shape = src.shape

    glacier_mask = rasterio.features.rasterize(
        [(geom, 1) for geom in rgi.geometry],
        out_shape=shape,
        transform=transform,
        fill=0,
        dtype=np.uint8,
    )

    valid = ~np.isnan(dem)

    # Elevation band boundaries: equal-area percentiles over valid pixels
    pct_bounds = np.linspace(0, 100, N_LAYERS + 1)
    elev_bounds = np.percentile(dem[valid], pct_bounds)

    fracs = np.zeros(N_LAYERS, dtype=np.float64)
    for k in range(N_LAYERS):
        lo, hi = elev_bounds[k], elev_bounds[k + 1]
        band = valid & (dem >= lo) & (dem < hi) if k < N_LAYERS - 1 else valid & (dem >= lo) & (dem <= hi)

        n_total = band.sum()
        n_glacier = (band & (glacier_mask == 1)).sum()
        fracs[k] = n_glacier / n_total if n_total > 0 else 0.0

    glacier_fractions[code] = fracs
    print(f"{cfg.name} glacier fractions: {[f'{f:.3f}' for f in fracs]}")


## Assemble `Catchment` Objects

All components are now ready. Each basin gets a `Catchment` instance:

```python
Catchment(
    mean_annual_solid_precip,  # mm/yr — from solid-precip partitioning
    n_layers,                  # number of elevation bands (N_LAYERS)
    hypsometric_curve,         # 101-element elevation percentile array
    input_elevation,           # gauge elevation from DEM
    glacier_fractions,         # per-band glacier coverage [0, 1]
)
```

Together with `Forcing` and `StreamflowSeries` built earlier, this is
everything the calibration step needs.

In [ ]:
catchments: dict[str, Catchment] = {}

for code, cfg in BASINS.items():
    catchment = Catchment(
        mean_annual_solid_precip=basin_data[code]["masp"],
        n_layers=N_LAYERS,
        hypsometric_curve=hypsometric_curves[code],
        input_elevation=gauge_elevations[code],
        glacier_fractions=glacier_fractions[code],
    )
    catchments[code] = catchment

    curve = hypsometric_curves[code]
    fracs = glacier_fractions[code]
    print(
        f"\n{cfg.name}:\n"
        f"  mean annual solid precip = {basin_data[code]['masp']:.0f} mm/yr\n"
        f"  n_layers = {N_LAYERS}\n"
        f"  elevation range = {curve[0]:.0f}–{curve[-1]:.0f} m\n"
        f"  gauge elevation = {gauge_elevations[code]:.0f} m\n"
        f"  max glacier fraction = {fracs.max():.3f} (band {fracs.argmax()})"
    )

## Selected Basin Calibration and Validation

The notebook below is parameterized from the config block at the top.

To switch basins, only change:
- `TARGET_BASIN_CODE`
- `TARGET_PERIOD_DEFINITIONS`

Everything else in the calibration and plotting workflow follows automatically.


In [ ]:
selected_cfg = BASINS[TARGET_BASIN_CODE]
selected_calibration_period = basin_data[TARGET_BASIN_CODE]["periods"]["calibration"]
selected_validation_period = basin_data[TARGET_BASIN_CODE]["periods"]["validation"]
selected_streamflow_source = basin_data[TARGET_BASIN_CODE]["streamflow_source"]
selected_metric_name = CALIBRATION_SETTINGS["objective"]
selected_metric_fn, _ = get_metric(selected_metric_name)

selected_result = gr6j_cemaneige_glacier.calibrate(
    forcing=selected_calibration_period["forcing"],
    observed=selected_calibration_period["observed"],
    catchment=catchments[TARGET_BASIN_CODE],
    **CALIBRATION_SETTINGS,
)
selected_validation_output = gr6j_cemaneige_glacier.run(
    selected_result.parameters,
    selected_validation_period["forcing"],
    catchment=catchments[TARGET_BASIN_CODE],
)

selected_calibration_simulated = selected_result.output.streamflow[CALIBRATION_SETTINGS["warmup"] :]
selected_validation_simulated = selected_validation_output.streamflow[CALIBRATION_SETTINGS["warmup"] :]

selected_calibration_score = selected_metric_fn(
    selected_calibration_period["observed"].streamflow,
    selected_calibration_simulated,
)
selected_validation_score = selected_metric_fn(
    selected_validation_period["observed"].streamflow,
    selected_validation_simulated,
)
selected_calibration_nse = nse(
    selected_calibration_period["observed"].streamflow,
    selected_calibration_simulated,
)
selected_validation_nse = nse(
    selected_validation_period["observed"].streamflow,
    selected_validation_simulated,
)
selected_calibration_kge = kge(
    selected_calibration_period["observed"].streamflow,
    selected_calibration_simulated,
)
selected_validation_kge = kge(
    selected_validation_period["observed"].streamflow,
    selected_validation_simulated,
)

print(f"Selected basin: {selected_cfg.name} ({TARGET_BASIN_CODE})")
print(f"Streamflow source: {selected_streamflow_source.name}")
print(f"Calibration period: {selected_calibration_period['obs_start'].date()} to {selected_calibration_period['obs_end'].date()}")
print(f"Validation period: {selected_validation_period['obs_start'].date()} to {selected_validation_period['obs_end'].date()}")
print()
print("Calibrated parameters:")
print(selected_result.parameters)
print()
print(f"Calibration {selected_metric_name.upper()}: {selected_calibration_score:.3f}")
print(f"Validation {selected_metric_name.upper()}: {selected_validation_score:.3f}")
if selected_metric_name != "nse":
    print(f"Calibration NSE: {selected_calibration_nse:.3f}")
    print(f"Validation NSE: {selected_validation_nse:.3f}")
if selected_metric_name != "kge":
    print(f"Calibration KGE: {selected_calibration_kge:.3f}")
    print(f"Validation KGE: {selected_validation_kge:.3f}")


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), constrained_layout=True, sharey=True)

plot_specs = [
    (
        axes[0],
        f"Calibration ({selected_calibration_period['obs_start'].year}-{selected_calibration_period['obs_end'].year})",
        selected_calibration_period,
        selected_calibration_simulated,
        selected_calibration_score,
        selected_calibration_nse,
        selected_calibration_kge,
    ),
    (
        axes[1],
        f"Validation ({selected_validation_period['obs_start'].year}-{selected_validation_period['obs_end'].year})",
        selected_validation_period,
        selected_validation_simulated,
        selected_validation_score,
        selected_validation_nse,
        selected_validation_kge,
    ),
]

for ax, title, period, simulated_streamflow, objective_score, _, _ in plot_specs:
    observed_time = pd.to_datetime(period["observed"].time)
    simulated_time = pd.to_datetime(period["forcing"].time[period["warmup_days"] :])

    ax.plot(observed_time, period["observed"].streamflow, color="#d95f02", linewidth=1.2, label="Observed")
    ax.plot(simulated_time, simulated_streamflow, color="#1b9e77", linewidth=1.0, label="Simulated")
    ax.set_title(f"{selected_cfg.name} {title} | {selected_metric_name.upper()} = {objective_score:.3f}")
    ax.set_ylabel("Streamflow [mm/day]")
    ax.grid(alpha=0.25)
    sns.despine(ax=ax)
    ax.legend()

axes[1].set_xlabel("Date")
plt.show()
